# 日米の高収益企業の特徴比較（MSCI World 構成銘柄）

**目的**：MSCI World（先進国）構成銘柄のうち、米国（US）と日本（JP）それぞれの「高収益企業」が
どのような特徴を持つのかを洗い出し、日米の違いを整理する。

**ノートブックの構成**（上から順に実行すれば完結します。pickle は使っていません）

| # | 内容 |
|---|---|
| 0 | 設定・パラメータ |
| 1 | データ取得（`get_profitability_data`） |
| 2 | 前処理（型変換・セクター付与・外れ値処理）とデータ概観 |
| 3 | 企業の分け方（国×月内パーセンタイルで 高・中・低収益 の3階層 ＋ 十分位） |
| 4 | 収益性水準の推移と分布 |
| 5 | 高・中・低収益企業の特徴プロファイル（自国内での相対位置, 高−低スプレッド） |
| 5-2 | 分位分析：収益性に沿って特徴が線形・単調に変化するか |
| 6 | ROEの源泉：DuPont分解（マージン × 回転率 × レバレッジ） |
| 7 | セクター構成 |
| 8 | 収益性の持続性（遷移・指数除外を含む行き先・何年続くか・残る企業の特徴・平均回帰の速さ） |
| 9 | 利益の質・予想収益性 |
| 10 | 株主還元（総還元：稼いだ利益をどれだけ株主に返しているか） |
| 11 | バリュエーション（市場は収益性を評価しているか） |
| 12 | 規模・成長・投資 |
| 13 | 多変量分析（Fama-MacBeth型の月次クロスセクション回帰） |
| 14 | 頑健性（定義・期間・階層の区切りの変更） |
| 15 | まとめ（自動抽出） |

**比較の考え方**
- 日米で ROE などの水準が大きく異なるため、収益性は **各国・各月のクロスセクション内でのパーセンタイル順位** で測る。
  - 3階層：下位30%＝低収益 / 中位40%＝中収益 / 上位30%＝高収益（区切りはパラメータで変更可）
  - 分位：十分位（Q1〜Q10）。分位に沿って特徴が線形・単調に変わるかを確認する
- 特徴量も **各国・各月内で標準化（zスコア）** し、「自国の平均的な企業と比べてどれだけ偏っているか」で比べる。
  これにより、会計基準や単位の違い・市場全体の水準差の影響を抑えて「高収益企業らしさ」の日米差を見る。
- 時系列の平均と Newey-West の t 値で、偶然でない偏りかを判断する。

## 0. 設定

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import japanize_matplotlib  # noqa: F401  日本語フォント設定

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COUNTRY_COLORS = {"US": "#1f4e9c", "JP": "#c0392b"}
COUNTRY_LABELS = {"US": "米国", "JP": "日本"}

In [ ]:
# ===== パラメータ =====
START_DATE = 200001          # 取得開始（YYYYMM）
END_DATE = 999999            # 取得終了（YYYYMM）

HIGH_PROF_VAR = "roe_actual"   # 高収益企業を定義する指標（roe_actual / roa_actual / profitability_factor など）
TIER_CUTS = (0.3, 0.7)         # パーセンタイルの区切り：〜30%=低収益 / 30〜70%=中収益 / 70%〜=高収益
N_QUANTILES = 10               # 分位分析の分位数（10=十分位, 5=五分位）

EXCLUDE_SECTOR_CODES = ["40"]  # 除外するGICSセクター（40=金融：ROE・レバレッジの構造が異なるため）。含める場合は []
WINSOR_LOWER, WINSOR_UPPER = 0.01, 0.99   # 国×月ごとの外れ値処理（分位点でクリップ）
MIN_SECTOR_N = 5               # セクター中立の順位付けに必要な最小銘柄数
NW_LAGS = 12                   # Newey-West t値のラグ（月次で重複する情報が多いため12）

# 期間分割（頑健性チェック用）。日本はアベノミクス／CGコード（2015）・東証PBR要請（2023）などを意識
PERIODS = {
    "2000-2012": (2000, 2012),
    "2013-2019": (2013, 2019),
    "2020-": (2020, 2100),
}

# True にすると DB に接続せず架空データで動作確認できる（分析結果には意味がない）
USE_DUMMY_DATA = False

## 1. データ取得

`fetch_data(QUERY, "irddb")` は社内DB接続用の関数です。普段お使いの import 文を下のセルに記入してください。

In [ ]:
# 社内DB接続関数の読み込み（環境に合わせて書き換えてください）
# from xxxxx import fetch_data

if not USE_DUMMY_DATA and "fetch_data" not in globals():
    raise NameError("fetch_data が定義されていません。上の import 文を環境に合わせて記入してください。")

In [ ]:
def get_profitability_data(country="US", start_date=200001, end_date=999999):
    country = country.strip().upper()

    if country not in {"US", "JP"}:
        raise ValueError("countryは'US'または'JP'を指定してください。")

    start_date = int(start_date)
    end_date = int(end_date)

    QUERY = f"""
    DECLARE @StartDate INT = {start_date};
    DECLARE @EndDate   INT = {end_date};

    SELECT
        -- 識別・管理
        F.[DATE]    AS [year_month],
        F.[GID]     AS [Global_ID],
        F.[BID]     AS [Barra_ID],
        F.[SEDOL]   AS [SEDOL],
        F.[UPDDATE] AS [update_date],

        -- MSCI銘柄情報
        NULLIF(M.[NAME], '-999999998')
            AS [msci_name],
        M.[CTRY]
            AS [msci_country],
        NULLIF(M.[GICS], '00000000')
            AS [gics_code],

        -- 企業規模
        NULLIF(F.[TMV_USD], -999999998)
            AS [company_market_cap_usd],
        NULLIF(F.[MV_USD], -999999998)
            AS [security_market_cap_usd],

        -- 実績収益性
        NULLIF(F.[ROE_ACT], -999999998)
            AS [roe_actual],
        NULLIF(F.[ROA_ACT], -999999998)
            AS [roa_actual],

        -- 予想収益性
        NULLIF(F.[ROE_EST], -999999998)
            AS [roe_estimate],
        NULLIF(F.[ROA_EST], -999999998)
            AS [roa_estimate],

        -- 収益性ファクター
        NULLIF(F.[PROF], -999999998)
            AS [profitability_factor],

        -- 収益性の源泉
        NULLIF(F.[GM_ACT], -999999998)
            AS [gross_margin_actual],
        NULLIF(F.[OIS_ACT], -999999998)
            AS [operating_margin_actual],
        NULLIF(F.[NIS_ACT], -999999998)
            AS [net_margin_actual],
        NULLIF(F.[TA_TO], -999999998)
            AS [asset_turnover_actual],

        -- 財務構成・利益の質
        NULLIF(F.[EQR], -999999998)
            AS [equity_ratio],
        NULLIF(F.[TD_TA], -999999998)
            AS [debt_to_assets],
        NULLIF(F.[OCF_TA], -999999998)
            AS [operating_cf_to_assets],

        -- バリュエーション
        NULLIF(F.[BP_ACT], -999999998)
            AS [book_to_price_actual],
        NULLIF(F.[EP_ACT], -999999998)
            AS [earnings_to_price_actual],

        -- 成長・投資
        NULLIF(F.[SALES_GRW], -999999998)
            AS [sales_growth_actual],
        NULLIF(F.[TA_GRW], -999999998)
            AS [asset_growth_actual],
        NULLIF(F.[IA], -999999998)
            AS [investment_to_assets_actual],

        -- 株主還元：配当
        NULLIF(F.[DP_ACT], -999999998)
            AS [dividend_yield_actual],
        NULLIF(F.[DP_EST], -999999998)
            AS [dividend_yield_estimate],
        NULLIF(F.[DOE], -999999998)
            AS [dividend_on_equity],

        -- 株主還元：自社株買い・総還元・純還元
        NULLIF(F.[BUYBACK], -999999998)
            AS [buyback_yield],
        NULLIF(F.[GPY], -999999998)
            AS [gross_payout_yield],
        NULLIF(F.[NPY], -999999998)
            AS [net_payout_yield]

    FROM [GLOBAL].[quants].[FACTOR] AS F

    INNER JOIN [GLOBAL].[quants].[MSCI] AS M
        ON M.[DATE] = F.[DATE]
        AND M.[GID]  = F.[GID]

    WHERE F.[DATE] BETWEEN @StartDate AND @EndDate
        AND F.[BID] <> 'XXXXXXX'
        AND M.[F_IDX_DM] = 1
        AND M.[CTRY] = '{country}'

    ORDER BY
        F.[DATE],
        F.[GID];
    """

    df = fetch_data(QUERY, "irddb")

    # 文字列の前後の空白を除去（pandas 2.1未満は applymap）
    strip = lambda x: x.strip() if isinstance(x, str) else x
    df = df.map(strip) if hasattr(df, "map") else df.applymap(strip)

    return df

In [ ]:
def make_dummy_data(country, start_date=200001, end_date=202412, n_firms=None, seed=0):
    """動作確認用の架空データ（USE_DUMMY_DATA=True のときのみ使用）。分析上の意味はない。"""
    rng = np.random.default_rng(seed + (0 if country == "US" else 1))
    n_firms = n_firms or (400 if country == "US" else 200)
    dates = pd.period_range(str(max(start_date, 200001))[:4] + "-" + str(max(start_date, 200001))[4:],
                            str(min(end_date, 202412))[:4] + "-" + str(min(end_date, 202412))[4:], freq="M")
    sectors = ["10102010", "15101010", "20101010", "25101010", "30101010", "35101010",
               "40101010", "45101010", "50101010", "55101010", "60101010"]
    firm_sector = rng.choice(sectors, n_firms)
    base_nm = rng.normal(0.08 if country == "US" else 0.05, 0.05, n_firms)
    base_to = rng.lognormal(np.log(0.7 if country == "US" else 0.8), 0.4, n_firms)
    base_eq = rng.uniform(0.2, 0.6 if country == "US" else 0.7, n_firms)
    rows = []
    nm, to, eq = base_nm.copy(), base_to.copy(), base_eq.copy()
    mcap = rng.lognormal(np.log(2e10 if country == "US" else 8e9), 1.0, n_firms)
    for d in dates:
        nm = 0.97 * nm + 0.03 * base_nm + rng.normal(0, 0.005, n_firms)
        to = to * np.exp(rng.normal(0, 0.02, n_firms))
        eq = np.clip(eq + rng.normal(0, 0.005, n_firms), 0.05, 0.95)
        mcap = mcap * np.exp(rng.normal(0.005, 0.07, n_firms))
        roa = nm * to
        roe = roa / eq
        df = pd.DataFrame({
            "year_month": int(d.strftime("%Y%m")),
            "Global_ID": [f"{country}{i:05d}" for i in range(n_firms)],
            "Barra_ID": [f"B{country}{i:04d}" for i in range(n_firms)],
            "SEDOL": None, "update_date": None,
            "msci_name": [f"{country} Corp {i}" for i in range(n_firms)],
            "msci_country": country, "gics_code": firm_sector,
            "company_market_cap_usd": mcap, "security_market_cap_usd": mcap,
            "roe_actual": roe, "roa_actual": roa,
            "roe_estimate": roe * 0.8 + 0.02 + rng.normal(0, 0.02, n_firms),
            "roa_estimate": roa * 0.8 + 0.01,
            "profitability_factor": (roe - 0.1) * 5 + rng.normal(0, 0.3, n_firms),
            "gross_margin_actual": nm + rng.uniform(0.1, 0.4, n_firms),
            "operating_margin_actual": nm * 1.4 + rng.normal(0, 0.01, n_firms),
            "net_margin_actual": nm, "asset_turnover_actual": to,
            "equity_ratio": eq, "debt_to_assets": np.clip(1 - eq - rng.uniform(0.1, 0.3, n_firms), 0, None),
            "operating_cf_to_assets": roa + rng.normal(0.03, 0.03, n_firms),
            "book_to_price_actual": rng.lognormal(np.log(0.4 if country == "US" else 0.9), 0.5, n_firms),
            "earnings_to_price_actual": rng.normal(0.05, 0.03, n_firms),
            "sales_growth_actual": rng.normal(0.05, 0.15, n_firms),
            "asset_growth_actual": rng.normal(0.05, 0.12, n_firms),
            "investment_to_assets_actual": rng.normal(0.05, 0.1, n_firms),
            "dividend_yield_actual": rng.uniform(0, 0.05, n_firms),
            "dividend_yield_estimate": rng.uniform(0, 0.05, n_firms),
            "dividend_on_equity": rng.uniform(0, 0.06, n_firms),
            "buyback_yield": rng.uniform(0, 0.04 if country == "US" else 0.015, n_firms),
            "gross_payout_yield": rng.uniform(0, 0.08, n_firms),
            "net_payout_yield": rng.uniform(-0.02, 0.07, n_firms),
        })
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def load_data(country):
    if USE_DUMMY_DATA:
        return make_dummy_data(country, START_DATE, END_DATE)
    return get_profitability_data(country, START_DATE, END_DATE)

In [ ]:
df_us = load_data("US")
df_jp = load_data("JP")

raw = pd.concat([df_us.assign(country="US"), df_jp.assign(country="JP")], ignore_index=True)
print(f"US: {len(df_us):,} 行 / JP: {len(df_jp):,} 行")
raw.head()

## 2. 前処理とデータ概観

- `year_month`（YYYYMM）を日付に変換
- 数値列を数値型へ（念のため欠損コード `-999999998` 系も欠損に）
- GICS コード先頭2桁からセクターを付与し、指定セクター（既定：金融）を除外
- 国×月ごとに 1%/99% でウィンザー化

In [ ]:
GICS_SECTOR = {
    "10": "エネルギー", "15": "素材", "20": "資本財・サービス", "25": "一般消費財・サービス",
    "30": "生活必需品", "35": "ヘルスケア", "40": "金融", "45": "情報技術",
    "50": "コミュニケーション・サービス", "55": "公益事業", "60": "不動産",
}

# 分析に使う数値列と日本語ラベル
LABELS = {
    "roe_actual": "ROE（実績）",
    "roa_actual": "ROA（実績）",
    "roe_estimate": "ROE（予想）",
    "roa_estimate": "ROA（予想）",
    "profitability_factor": "収益性ファクター",
    "gross_margin_actual": "売上総利益率",
    "operating_margin_actual": "営業利益率",
    "net_margin_actual": "純利益率",
    "asset_turnover_actual": "総資産回転率",
    "equity_ratio": "自己資本比率",
    "debt_to_assets": "有利子負債/総資産",
    "operating_cf_to_assets": "営業CF/総資産",
    "book_to_price_actual": "B/P",
    "earnings_to_price_actual": "E/P",
    "sales_growth_actual": "売上成長率",
    "asset_growth_actual": "総資産成長率",
    "investment_to_assets_actual": "投資/総資産",
    "dividend_yield_actual": "配当利回り（実績）",
    "dividend_yield_estimate": "配当利回り（予想）",
    "dividend_on_equity": "DOE",
    "buyback_yield": "自社株買い利回り",
    "gross_payout_yield": "総還元利回り",
    "net_payout_yield": "純還元利回り",
    "company_market_cap_usd": "時価総額（企業, USD）",
    "security_market_cap_usd": "時価総額（銘柄, USD）",
    # 派生変数
    "log_mcap": "log時価総額",
    "total_payout_ratio": "総還元性向（総還元利回り÷E/P）",
    "net_payout_ratio": "純還元性向（純還元利回り÷E/P）",
    "accrual": "アクルーアル（ROA−営業CF/総資産）",
    "roe_change_est": "予想ROE−実績ROE",
}
NUM_COLS = [c for c in LABELS if c in raw.columns]
lab = lambda c: LABELS.get(c, c)

In [ ]:
df = raw.copy()
df["date"] = pd.to_datetime(df["year_month"].astype(int).astype(str), format="%Y%m")

for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[NUM_COLS] = df[NUM_COLS].mask(df[NUM_COLS] <= -999999990)   # 欠損コードの取りこぼし対策

# GICS（数値で返ってきても文字列で返ってきても対応）
gics = pd.to_numeric(df["gics_code"], errors="coerce").astype("Int64").astype("string")
df["sector_code"] = gics.str[:2]
df["sector"] = df["sector_code"].map(GICS_SECTOR).fillna("不明")

# 重複（同一国・同一月・同一GID）は最初の1行を残す
n0 = len(df)
df = df.drop_duplicates(["country", "date", "Global_ID"]).copy()
print(f"重複除去: {n0 - len(df):,} 行")

# セクター除外
n0 = len(df)
df = df[~df["sector_code"].isin(EXCLUDE_SECTOR_CODES)].copy()
print(f"セクター除外 {EXCLUDE_SECTOR_CODES}: {n0 - len(df):,} 行")

# 派生変数
df["log_mcap"] = np.log(df["company_market_cap_usd"].where(df["company_market_cap_usd"] > 0))
df["accrual"] = df["roa_actual"] - df["operating_cf_to_assets"]
df["roe_change_est"] = df["roe_estimate"] - df["roe_actual"]
# 総還元性向 = (配当+自社株買い)/純利益 = 総還元利回り / 益利回り（E/P>0 の企業のみ。赤字企業は定義できない）
ep_pos = df["earnings_to_price_actual"].where(df["earnings_to_price_actual"] > 0)
df["total_payout_ratio"] = df["gross_payout_yield"] / ep_pos
df["net_payout_ratio"] = df["net_payout_yield"] / ep_pos

ANALYSIS_COLS = [c for c in NUM_COLS + ["log_mcap", "accrual", "roe_change_est",
                                        "total_payout_ratio", "net_payout_ratio"] if c in df.columns]

In [ ]:
def winsorize_by_group(data, cols, keys, lower=0.01, upper=0.99):
    """keys（国×月など）ごとに分位点でクリップする。"""
    g = data.groupby(keys)[cols]
    lo = data[keys].join(g.quantile(lower), on=keys)[cols]
    hi = data[keys].join(g.quantile(upper), on=keys)[cols]
    out = data.copy()
    out[cols] = data[cols].clip(lower=lo, upper=hi)
    return out


df = winsorize_by_group(df, ANALYSIS_COLS, ["country", "date"], WINSOR_LOWER, WINSOR_UPPER)
df = df.sort_values(["country", "date", "Global_ID"]).reset_index(drop=True)
print(f"分析対象: {len(df):,} 行, 期間 {df['date'].min():%Y-%m} 〜 {df['date'].max():%Y-%m}")

### 単位の確認
DB の値が「%表記」か「小数表記」かで解釈が変わるため、まず国別の中央値を確認します
（以降の分析は国内標準化・順位・対数差を中心にしているので単位に依存しにくい設計です）。

In [ ]:
summary_stats = (
    df.groupby("country")[ANALYSIS_COLS]
    .median()
    .T.rename(index=LABELS)
    .rename(columns=COUNTRY_LABELS)
)
summary_stats.columns.name = "中央値（全期間プール）"
summary_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cnt = df.groupby(["date", "country"])["Global_ID"].nunique().unstack()
for c in ["US", "JP"]:
    axes[0].plot(cnt.index, cnt[c], color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].set_title("銘柄数の推移（金融除外後）")
axes[0].legend()

miss = df.groupby("country")[ANALYSIS_COLS].apply(lambda x: x.isna().mean()).T
miss = miss.rename(index=LABELS)
x = np.arange(len(miss))
axes[1].bar(x - 0.2, miss["US"], 0.4, color=COUNTRY_COLORS["US"], label="米国")
axes[1].bar(x + 0.2, miss["JP"], 0.4, color=COUNTRY_COLORS["JP"], label="日本")
axes[1].set_xticks(x)
axes[1].set_xticklabels(miss.index, rotation=90, fontsize=8)
axes[1].set_title("欠損率（全期間）")
axes[1].legend()
plt.tight_layout()
plt.show()

sec_cnt = (df.groupby(["country", "sector"])["Global_ID"].nunique().unstack(0)
           .rename(columns=COUNTRY_LABELS).fillna(0).astype(int))
sec_cnt.columns.name = "ユニーク銘柄数"
sec_cnt

## 3. 収益性による企業の分け方

各国・各月のクロスセクションで `HIGH_PROF_VAR`（既定：ROE実績）の **パーセンタイル順位**（0〜1）を計算し、2通りに区切ります。

| 区分 | 定義（既定） | 用途 |
|---|---|---|
| **3階層** | 下位30% = 低収益 / 中位40% = 中収益 / 上位30% = 高収益（`TIER_CUTS`） | 高・中・低の特徴比較、高−低スプレッド |
| **分位** | 十分位 Q1〜Q10（`N_QUANTILES`） | 分位に沿って特徴が線形・単調に変化するかのチェック |

- 順位は国内で付けるため、日米で ROE の水準が違っても「その国の中での相対的な位置」で比べられる
- 補助として、セクター内順位（業種構成の違いを除く）と日米合算順位（絶対水準での比較）も作る

In [ ]:
TIERS = ["低収益", "中収益", "高収益"]
TIER_STYLE = {"高収益": "-", "中収益": "-.", "低収益": ":"}
TIER_COLORS = {"高収益": "#2a9d8f", "中収益": "#e9c46a", "低収益": "#9e9e9e"}
QLABELS = [f"Q{i}" for i in range(1, N_QUANTILES + 1)]
TIER_DESC = (f"低収益=下位{TIER_CUTS[0]:.0%} / 中収益={TIER_CUTS[0]:.0%}〜{TIER_CUTS[1]:.0%} / "
             f"高収益=上位{1 - TIER_CUTS[1]:.0%}")


def pct_rank(data, var, keys, min_n=1):
    """keys 内でのパーセンタイル順位（0〜1, 大きいほど高収益）"""
    r = data.groupby(keys)[var].rank(pct=True)
    n = data.groupby(keys)[var].transform("count")
    return r.where(n >= min_n)


def to_tier(p):
    t = pd.cut(p, [0, TIER_CUTS[0], TIER_CUTS[1], 1.0], labels=TIERS, include_lowest=True)
    return t.astype("object").where(p.notna())


def to_quantile(p, n=N_QUANTILES):
    return np.ceil(p * n).clip(1, n)


def add_group_columns(data, var):
    out = data.copy()
    out["pct"] = pct_rank(out, var, ["country", "date"])
    out["tier"] = to_tier(out["pct"])
    out["q"] = to_quantile(out["pct"])
    out["pct_sector"] = pct_rank(out, var, ["country", "date", "sector"], min_n=MIN_SECTOR_N)
    out["tier_sector"] = to_tier(out["pct_sector"])
    out["pct_global"] = pct_rank(out, var, ["date"])
    out["tier_global"] = to_tier(out["pct_global"])
    out["is_high"] = (out["tier"] == "高収益").where(out["tier"].notna())
    out["is_low"] = (out["tier"] == "低収益").where(out["tier"].notna())
    return out


df = add_group_columns(df, HIGH_PROF_VAR)
print(TIER_DESC)
(df.groupby(["country", "tier"])["Global_ID"].count().unstack()[TIERS]
 .div(df.groupby("country")["tier"].count(), axis=0).rename(index=COUNTRY_LABELS))

In [ ]:
# 各階層の境界値（30%点・70%点）と階層別中央値の推移：日本の「高収益」は米国の何%水準か
cuts = (df.dropna(subset=["tier"]).groupby(["date", "country"])[HIGH_PROF_VAR]
        .quantile(list(TIER_CUTS)).unstack([1, 2]))
tier_med = df.dropna(subset=["tier"]).groupby(["date", "country", "tier"])[HIGH_PROF_VAR].median().unstack([1, 2])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, c in zip(axes, ["US", "JP"]):
    for t in TIERS:
        ax.plot(tier_med.index, tier_med[(c, t)], color=TIER_COLORS[t], label=f"{t}の中央値")
    for k in TIER_CUTS:
        ax.plot(cuts.index, cuts[(c, k)], color="black", ls="--", lw=0.8)
    ax.set_title(f"{COUNTRY_LABELS[c]}：{lab(HIGH_PROF_VAR)}（破線は{TIER_CUTS[0]:.0%}/{TIER_CUTS[1]:.0%}境界）")
    ax.legend()
plt.tight_layout()
plt.show()

cut_tbl = cuts.mean().unstack(0).rename(columns=COUNTRY_LABELS)
cut_tbl.index = [f"{k:.0%}点" for k in cut_tbl.index]
cut_tbl

In [ ]:
# 日米合算で順位付けした場合（絶対水準）：各国の企業がどの階層に入るか
glob = (df.dropna(subset=["tier_global"]).groupby(["date", "country"])["tier_global"]
        .value_counts(normalize=True).unstack())
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), sharey=True)
for ax, c in zip(axes, ["US", "JP"]):
    g = glob.xs(c, level="country")
    ax.stackplot(g.index, [g[t] for t in TIERS], colors=[TIER_COLORS[t] for t in TIERS], labels=TIERS)
    ax.set_title(f"{COUNTRY_LABELS[c]}企業の日米共通基準での階層構成")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.set_ylim(0, 1)
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

## 4. 収益性水準の推移と分布

In [ ]:
def plot_country_median_ts(data, cols, ncols=3, title=None):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.3 * nrows), squeeze=False)
    med = data.groupby(["date", "country"])[cols].median()
    for ax, col in zip(axes.flat, cols):
        m = med[col].unstack()
        for c in ["US", "JP"]:
            if c in m:
                ax.plot(m.index, m[c], color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
        ax.set_title(lab(col))
    for ax in axes.flat[len(cols):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


plot_country_median_ts(
    df,
    ["roe_actual", "roa_actual", "profitability_factor",
     "operating_margin_actual", "net_margin_actual", "asset_turnover_actual"],
    title="収益性指標の中央値の推移",
)

In [ ]:
# 直近時点の分布
latest = df["date"].max()
snap = df[df["date"] == latest]
cols = ["roe_actual", "roa_actual", "operating_margin_actual", "asset_turnover_actual", "equity_ratio", "book_to_price_actual"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, cols):
    for c in ["US", "JP"]:
        v = snap.loc[snap["country"] == c, col].dropna()
        ax.hist(v, bins=40, alpha=0.5, density=True, color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]} (n={len(v)})")
        ax.axvline(v.median(), color=COUNTRY_COLORS[c], ls="--", lw=1)
    ax.set_title(lab(col))
axes.flat[0].legend()
fig.suptitle(f"分布の比較（{latest:%Y-%m}時点、破線は中央値）", y=1.01)
plt.tight_layout()
plt.show()

## 5. 高・中・低収益企業の特徴プロファイル

各国・各月で特徴量を標準化（zスコア）し、階層ごとの平均zスコアを月ごとに計算 → 全期間で平均します。

- **高−低スプレッド**（高収益の平均z − 低収益の平均z）が、その特徴と収益性の結び付きの強さ
- t値は Newey-West（ラグ12）。|t| > 2 程度を目安に「一貫した差」とみなす
- スプレッドの日米差（JP − US）が大きい項目が、**日米で高収益企業の性格が異なるポイント**

※ ROE を定義に使っているので ROE・ROA・マージン等の差は当然大きい。注目すべきは
**財務構成・還元・バリュエーション・成長・規模** といった「定義に直接含まれない」項目と、ROEの構成要素のバランス。

In [ ]:
PROFILE_FEATS = [c for c in [
    "roe_actual", "roa_actual", "roe_estimate", "profitability_factor",
    "gross_margin_actual", "operating_margin_actual", "net_margin_actual", "asset_turnover_actual",
    "equity_ratio", "debt_to_assets", "operating_cf_to_assets", "accrual",
    "book_to_price_actual", "earnings_to_price_actual",
    "sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual",
    "gross_payout_yield", "net_payout_yield", "total_payout_ratio", "net_payout_ratio",
    "log_mcap",
] if c in df.columns]

FEATURE_BLOCK = {
    "収益性": ["roe_actual", "roa_actual", "roe_estimate", "profitability_factor"],
    "収益の源泉": ["gross_margin_actual", "operating_margin_actual", "net_margin_actual", "asset_turnover_actual"],
    "財務構成・利益の質": ["equity_ratio", "debt_to_assets", "operating_cf_to_assets", "accrual"],
    "バリュエーション": ["book_to_price_actual", "earnings_to_price_actual"],
    "成長・投資": ["sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual"],
    "株主還元（総還元）": ["gross_payout_yield", "net_payout_yield", "total_payout_ratio", "net_payout_ratio"],
    "規模": ["log_mcap"],
}
DEF_FEATS = [c for c in FEATURE_BLOCK["収益性"] if c in PROFILE_FEATS]   # 定義に近い項目（比較から外すことがある）


def nw_tstat(x, lags=NW_LAGS):
    """時系列平均の Newey-West t値"""
    x = pd.Series(x).dropna().to_numpy()
    n = len(x)
    if n < 3:
        return np.nan
    e = x - x.mean()
    s = e @ e / n
    for l in range(1, min(lags, n - 1) + 1):
        s += 2 * (1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n
    return x.mean() / np.sqrt(s / n) if s > 0 else np.nan


def cs_zscore(data, cols, keys=("country", "date")):
    keys = list(keys)
    g = data.groupby(keys)[cols]
    return (data[cols] - g.transform("mean")) / g.transform("std")


def group_mean_z(data, group_col, feats=PROFILE_FEATS):
    """国×月×グループごとの平均zスコア（時系列）"""
    z = cs_zscore(data, feats)
    z[["country", "date"]] = data[["country", "date"]]
    z["grp"] = data[group_col]
    return z.dropna(subset=["grp"]).groupby(["country", "date", "grp"])[feats].mean()


def tier_profile(data, tier_col="tier", feats=PROFILE_FEATS):
    """階層別の平均z（全期間平均）と高−低スプレッド・t値"""
    ts = group_mean_z(data, tier_col, feats)
    tier_mean = ts.groupby(["country", "grp"]).mean()
    spread_ts = ts.xs("高収益", level="grp") - ts.xs("低収益", level="grp")
    res = pd.DataFrame(index=feats)
    for c in ["US", "JP"]:
        for t in TIERS:
            res[f"{c}_{t}"] = tier_mean.loc[(c, t)]
        res[f"{c}_高−低"] = spread_ts.loc[c].mean()
        res[f"{c}_t"] = spread_ts.loc[c].agg(nw_tstat)
    res["JP−US"] = res["JP_高−低"] - res["US_高−低"]
    return res, spread_ts


profile, spread_ts = tier_profile(df)
show_cols = ["US_低収益", "US_中収益", "US_高収益", "US_高−低", "US_t",
             "JP_低収益", "JP_中収益", "JP_高収益", "JP_高−低", "JP_t", "JP−US"]
profile[show_cols].rename(index=LABELS).style.format("{:.2f}").background_gradient(
    cmap="RdBu_r", subset=[c for c in show_cols if not c.endswith("_t")], vmin=-1.5, vmax=1.5)

In [ ]:
def heatmap(mat, title, vmax=1.0, fmt="{:.2f}", figsize=None, xrot=30):
    figsize = figsize or (1.2 * mat.shape[1] + 4, 0.38 * mat.shape[0] + 1.6)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mat.values.astype(float), cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8,
                        color="white" if abs(v) > 0.6 * vmax else "black")
    ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=xrot, ha="right" if xrot else "center")
    ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels([lab(i) for i in mat.index])
    ax.set_title(title)
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()


# 低→中→高 の並びで見たヒートマップ（各国の左から右へ色が素直に変われば単調）
ladder = profile[[f"{c}_{t}" for c in ["US", "JP"] for t in TIERS]].copy()
ladder.columns = [f"{COUNTRY_LABELS[c]}\n{t}" for c in ["US", "JP"] for t in TIERS]
heatmap(ladder, f"階層別の平均zスコア（{TIER_DESC}）", vmax=1.2, xrot=0)

In [ ]:
def plot_spread_bars(res, title, drop=DEF_FEATS):
    r = res.drop(index=[d for d in drop if d in res.index]).sort_values("JP−US")
    y = np.arange(len(r))
    fig, ax = plt.subplots(figsize=(9, 0.38 * len(r) + 1.5))
    for c, off in (("US", 0.2), ("JP", -0.2)):
        bars = ax.barh(y + off, r[f"{c}_高−低"], 0.4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
        for b, t in zip(bars, r[f"{c}_t"]):
            if abs(t) < 2:   # 有意でないものは薄く
                b.set_alpha(0.35)
    ax.set_yticks(y)
    ax.set_yticklabels([lab(i) for i in r.index])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("高収益 − 低収益 の平均zスコア差（自国内で標準化）")
    ax.set_title(title + "\n（薄い棒は|t|<2）")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


plot_spread_bars(profile, f"高収益企業と低収益企業の違い：日米比較（{lab(HIGH_PROF_VAR)}, JP−US差の順）")

In [ ]:
# 差の差（JP − US）：右ほど「日本でより強く収益性と結び付く特徴」
r = profile.drop(index=DEF_FEATS).sort_values("JP−US")
fig, ax = plt.subplots(figsize=(9, 0.35 * len(r) + 1.5))
ax.barh([lab(i) for i in r.index], r["JP−US"],
        color=np.where(r["JP−US"] > 0, COUNTRY_COLORS["JP"], COUNTRY_COLORS["US"]))
ax.axvline(0, color="black", lw=0.8)
ax.set_title("高−低スプレッドの日米差（JP−US）\n赤：日本で収益性とより強く結び付く / 青：米国でより強く結び付く")
plt.tight_layout()
plt.show()

In [ ]:
# スプレッドは時間とともに変化しているか（主要項目、12か月移動平均）
ts_cols = [c for c in ["equity_ratio", "asset_turnover_actual", "operating_margin_actual",
                       "book_to_price_actual", "total_payout_ratio", "log_mcap"] if c in spread_ts.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, col in zip(axes.flat, ts_cols):
    for c in ["US", "JP"]:
        s = spread_ts.loc[c, col].rolling(12, min_periods=6).mean()
        ax.plot(s.index, s, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{lab(col)}：高−低（z, 12M平均）")
axes.flat[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 絶対水準：階層別の中央値（月次中央値の時系列平均）
level = (df.dropna(subset=["tier"])
         .groupby(["country", "date", "tier"])[PROFILE_FEATS].median()
         .groupby(["country", "tier"]).mean().T)
level = level[[(c, t) for c in ["US", "JP"] for t in TIERS]]
level.index = [lab(i) for i in level.index]
level.columns = [f"{COUNTRY_LABELS[c]}:{t}" for c, t in level.columns]
level

## 5-2. 分位分析：収益性に沿って特徴は線形・単調に変化するか

十分位（Q1=最低収益 〜 Q10=最高収益）ごとに特徴量の平均zスコアを計算し、分位に沿った形を判定します。

| 指標 | 意味 |
|---|---|
| **傾き** | 分位を Q1→Q10 で線形回帰した傾き（Q1からQ10へ何σ動くかに相当）。月次で推計し時系列平均・NW t値 |
| **曲率** | 2次項の係数。有意なら直線では表せない（上位で加速 / 頭打ち / U字） |
| **スピアマン** | 分位と平均zの順位相関（月次の平均）。±1に近いほど単調 |
| **単調性** | 時系列平均したプロファイルで、隣り合う分位の変化が全体の方向と一致する割合 |
| **R²(線形)** | 時系列平均したプロファイルを直線で当てはめたときの決定係数 |

これにより「高収益企業だけが特殊なのか（上位で加速）」「収益性に比例して徐々に変わるのか（線形）」「中位が最も高い（逆U字）」などを区別できます。

In [ ]:
quant_ts = group_mean_z(df, "q")    # (country, date, 分位) × 特徴量 の平均z


def quantile_shape_stats(ts, feats=PROFILE_FEATS, min_months=24):
    qs = np.array(sorted(ts.index.get_level_values("grp").unique()))
    x = (qs - qs.mean()) / (qs.max() - qs.min())           # -0.5〜0.5（傾き = Q1→QNの変化量）
    x2 = x ** 2 - (x ** 2).mean()
    P1 = np.linalg.pinv(np.column_stack([np.ones_like(x), x]))
    P2 = np.linalg.pinv(np.column_stack([np.ones_like(x), x, x2]))
    xr = np.arange(len(qs)) - (len(qs) - 1) / 2
    rows = []
    for c in ["US", "JP"]:
        for f in feats:
            Y = ts.loc[c, f].unstack("grp").reindex(columns=qs).dropna()
            if len(Y) < min_months:
                continue
            V = Y.to_numpy()
            slope = V @ P1[1]
            curv = V @ P2[2]
            R = Y.rank(axis=1).to_numpy()
            Rc = R - R.mean(axis=1, keepdims=True)
            rho = (Rc @ xr) / (np.sqrt((Rc ** 2).sum(axis=1)) * np.sqrt((xr ** 2).sum()) + 1e-12)
            avg = Y.mean().to_numpy()
            direction = np.sign(avg[-1] - avg[0])
            mono = (np.sign(np.diff(avg)) == direction).mean() if direction != 0 else 0.0
            fit = np.column_stack([np.ones_like(x), x]) @ (P1 @ avg)
            r2 = 1 - ((avg - fit) ** 2).sum() / ((avg - avg.mean()) ** 2).sum()
            rows.append({
                "country": c, "feature": f,
                "傾き": slope.mean(), "傾きt": nw_tstat(slope),
                "曲率": curv.mean(), "曲率t": nw_tstat(curv),
                "スピアマン": rho.mean(), "単調性": mono, "R²(線形)": r2,
                f"Q{N_QUANTILES}−Q1": avg[-1] - avg[0],
            })
    return pd.DataFrame(rows).set_index(["feature", "country"])


def classify_shape(r):
    s, st, cv, ct, mono = r["傾き"], r["傾きt"], r["曲率"], r["曲率t"], r["単調性"]
    if abs(st) < 2 and abs(ct) < 2:
        return "関係なし"
    if mono >= 0.75 and abs(st) >= 2:
        d = "増加" if s > 0 else "減少"
        if abs(ct) < 2 or r["R²(線形)"] >= 0.95:
            return f"線形に{d}"
        return f"単調{d}・上位で加速" if np.sign(cv) == np.sign(s) else f"単調{d}・上位で頭打ち"
    if abs(ct) >= 2:
        return "U字（両端が高い）" if cv > 0 else "逆U字（中位が高い）"
    return "非単調（不規則）"


shape = quantile_shape_stats(quant_ts)
shape["形状"] = shape.apply(classify_shape, axis=1)
shape_wide = shape["形状"].unstack("country")[["US", "JP"]].rename(columns=COUNTRY_LABELS)
shape_wide["日米で形状が異なる"] = np.where(shape_wide["米国"] != shape_wide["日本"], "◯", "")
shape_wide.index = [lab(i) for i in shape_wide.index]
shape_wide

In [ ]:
# 詳細な統計量
shape_tbl = shape.drop(columns="形状").unstack("country")
shape_tbl = shape_tbl.reindex(columns=["US", "JP"], level=1)
shape_tbl.index = [lab(i) for i in shape_tbl.index]
shape_tbl.style.format("{:.2f}")

In [ ]:
def plot_quantile_z(ts, feats, title, ncols=4):
    feats = [f for f in feats if f in ts.columns]
    avg = ts.groupby(["country", "grp"]).mean()
    nrows = int(np.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.1 * nrows), squeeze=False)
    for ax, f in zip(axes.flat, feats):
        for c in ["US", "JP"]:
            s = avg.loc[c, f]
            ax.plot(s.index, s.values, marker="o", ms=4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
            b = np.polyfit(s.index.to_numpy(float), s.to_numpy(), 1)          # 線形当てはめ（破線）
            ax.plot(s.index, np.polyval(b, s.index.to_numpy(float)), color=COUNTRY_COLORS[c], ls="--", lw=0.8)
        for t0 in TIER_CUTS:                                                    # 3階層の境界
            ax.axvline(t0 * N_QUANTILES + 0.5, color="gray", lw=0.6, ls=":")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_xticks(range(1, N_QUANTILES + 1))
        ax.set_xticklabels(range(1, N_QUANTILES + 1), fontsize=8)
        sh = shape["形状"] if "shape" in globals() else None
        sub = " / ".join(f"{COUNTRY_LABELS[c][0]}:{sh.get((f, c), '')}" for c in ["US", "JP"]) if sh is not None else ""
        ax.set_title(f"{lab(f)}\n{sub}", fontsize=9)
    for ax in axes.flat[len(feats):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"【{title}】{lab(HIGH_PROF_VAR)}分位（1=低→{N_QUANTILES}=高）別の平均z（破線=線形当てはめ, 点線=3階層の境界）", y=1.01)
    plt.tight_layout()
    plt.show()


for block, feats in FEATURE_BLOCK.items():
    plot_quantile_z(quant_ts, feats, block)

In [ ]:
# 単調性・線形性の日米比較（スピアマンと R²）
fig, axes = plt.subplots(1, 2, figsize=(15, 0.35 * len(PROFILE_FEATS) + 1.8), sharey=True)
order = shape["スピアマン"].unstack("country").sort_values("US").index
y = np.arange(len(order))
for ax, col, xl in zip(axes, ["スピアマン", "R²(線形)"], [(-1, 1), (0, 1)]):
    v = shape[col].unstack("country").reindex(order)
    for c, off in (("US", 0.2), ("JP", -0.2)):
        ax.barh(y + off, v[c], 0.4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    ax.set_xlim(*xl)
    ax.set_title(col)
    ax.axvline(0, color="black", lw=0.8)
axes[0].set_yticks(y)
axes[0].set_yticklabels([lab(i) for i in order])
axes[0].legend()
fig.suptitle("分位に沿った単調性（スピアマン）と線形性（R²）", y=1.01)
plt.tight_layout()
plt.show()

## 6. ROE の源泉：DuPont 分解

$$\text{ROE} = \underbrace{\frac{\text{純利益}}{\text{売上}}}_{\text{純利益率}} \times \underbrace{\frac{\text{売上}}{\text{総資産}}}_{\text{回転率}} \times \underbrace{\frac{\text{総資産}}{\text{自己資本}}}_{\text{レバレッジ = 1/自己資本比率}}$$

対数を取ると和になるので、各国・各月のクロスセクションで

$$\mathrm{Var}(\log \text{ROE}) \approx \mathrm{Cov}(\log \text{ROE}, \log \text{NM}) + \mathrm{Cov}(\log \text{ROE}, \log \text{TO}) + \mathrm{Cov}(\log \text{ROE}, \log \text{LEV})$$

と分解し、**「ROE の企業間格差のうち何割がマージン/回転率/レバレッジで説明されるか」** を日米で比べます。
対数を使うため単位（%か小数か）に依存しません。ROE・純利益率などが正の企業のみが対象です（低収益の下位分位は赤字企業が多く、対象が少なくなる点に注意）。

In [ ]:
dp = df[["country", "date", "Global_ID", "q", "tier", "roe_actual", "net_margin_actual",
         "asset_turnover_actual", "equity_ratio"]].copy()
pos = (dp[["roe_actual", "net_margin_actual", "asset_turnover_actual", "equity_ratio"]] > 0).all(axis=1)
dp = dp[pos].copy()
dp["l_roe"] = np.log(dp["roe_actual"])
dp["l_nm"] = np.log(dp["net_margin_actual"])
dp["l_to"] = np.log(dp["asset_turnover_actual"])
dp["l_lev"] = -np.log(dp["equity_ratio"])
COMP = {"l_nm": "純利益率", "l_to": "総資産回転率", "l_lev": "レバレッジ"}

# 分解の当てはまり（定義が一致していれば相関≒1）
dp["l_sum"] = dp[list(COMP)].sum(axis=1)
fit = dp.groupby("country")[["l_roe", "l_sum"]].corr().xs("l_roe", level=1)["l_sum"]
print("log ROE と log(NM×TO×LEV) の相関:", fit.round(3).to_dict(),
      "（1から大きく離れる場合、ROEの定義（平均資本など）が構成要素と異なる）")

# 国×月ごとの分散分解
g = dp.groupby(["country", "date"])
var_roe = g["l_roe"].var()
shares = pd.DataFrame({
    name: g.apply(lambda x, c=c: x["l_roe"].cov(x[c])) / var_roe for c, name in COMP.items()
})
shares["その他（近似誤差）"] = 1 - shares.sum(axis=1)
share_mean = shares.groupby("country").mean().rename(index=COUNTRY_LABELS)
share_mean

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, 2]})

colors = ["#2a9d8f", "#e9c46a", "#e76f51", "#bbbbbb"]
left = np.zeros(len(share_mean))
for col, colr in zip(share_mean.columns, colors):
    axes[0].barh(share_mean.index, share_mean[col], left=left, color=colr, label=col)
    left += share_mean[col].values
axes[0].set_title("ROE格差の寄与度（全期間平均）")
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend(fontsize=8, loc="lower right")

for c in ["US", "JP"]:
    for name, ls in zip(COMP.values(), ["-", "--", ":"]):
        s = shares.loc[c, name].rolling(12, min_periods=6).mean()
        axes[1].plot(s.index, s, color=COUNTRY_COLORS[c], ls=ls, label=f"{COUNTRY_LABELS[c]}:{name}")
axes[1].set_title("寄与度の推移（12M平均）")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[1].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 分位別の DuPont 構成要素：各国・各月で中央分位の企業を基準(=1倍)にした幾何平均の倍率
#   低位分位は黒字企業が少ないため、国×月×分位で10社未満のセルは除外
gq = dp.dropna(subset=["q"]).groupby(["country", "date", "q"])
comp_q = gq[list(COMP)].mean()[gq.size() >= 10]
comp_q = comp_q - comp_q.groupby(["country", "date"]).transform("mean")     # 月内で中心化
comp_q = np.exp(comp_q.groupby(["country", "q"]).mean())

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (k, name) in zip(axes, COMP.items()):
    for c in ["US", "JP"]:
        s = comp_q.loc[c, k]
        ax.plot(s.index, s.values, marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    ax.axhline(1, color="black", lw=0.8)
    ax.set_yscale("log")
    ax.set_xticks(range(1, N_QUANTILES + 1))
    ax.set_title(f"{name}（分位平均=1倍）")
    ax.set_xlabel(f"{lab(HIGH_PROF_VAR)}分位")
axes[0].legend()
fig.suptitle("収益性分位ごとの DuPont 構成要素：上位ほどどの要素が伸びるか", y=1.02)
plt.tight_layout()
plt.show()

# 高収益 ÷ 低収益（黒字企業内）の倍率
tier_c = dp.dropna(subset=["tier"]).groupby(["country", "date", "tier"])[list(COMP)].mean().unstack("tier")
ratio = pd.concat({
    "高収益÷低収益": np.exp((tier_c.xs("高収益", axis=1, level=1) - tier_c.xs("低収益", axis=1, level=1)).groupby("country").mean()),
    "高収益÷中収益": np.exp((tier_c.xs("高収益", axis=1, level=1) - tier_c.xs("中収益", axis=1, level=1)).groupby("country").mean()),
}).rename(columns=COMP).rename(index=COUNTRY_LABELS, level=1)
ratio

In [ ]:
# マージン × 回転率 マップ（直近時点）
snap = df[(df["date"] == latest)].dropna(subset=["operating_margin_actual", "asset_turnover_actual", "tier"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for ax, c in zip(axes, ["US", "JP"]):
    s = snap[snap["country"] == c]
    for t in TIERS:
        h = s[s["tier"] == t]
        ax.scatter(h["asset_turnover_actual"], h["operating_margin_actual"], s=12, alpha=0.8,
                   color=TIER_COLORS[t], label=t)
    ax.set_xscale("log")
    ax.set_title(f"{COUNTRY_LABELS[c]}（{latest:%Y-%m}）")
    ax.set_xlabel("総資産回転率（対数軸）")
    ax.set_ylabel("営業利益率")
    ax.legend()
fig.suptitle("営業利益率 × 総資産回転率：収益階層の位置", y=1.01)
plt.tight_layout()
plt.show()

## 7. セクター構成

高収益企業・低収益企業がどの業種に偏っているかを日米で比較します。

In [ ]:
sec = df.dropna(subset=["tier"]).groupby(["country", "date", "sector"])["tier"].value_counts().unstack().fillna(0)
sec["全体"] = sec[TIERS].sum(axis=1)
sec_share = sec / sec.groupby(["country", "date"]).transform("sum")          # 各階層内の構成比
sec_hit = sec[TIERS].div(sec["全体"], axis=0)                                # セクター内で各階層に入る確率
sec_share = sec_share.groupby(["country", "sector"]).mean()
sec_hit = sec_hit.groupby(["country", "sector"]).mean()

sectors = sec_hit.xs("US", level="country")["高収益"].sort_values().index.tolist()
sectors += [s for s in df["sector"].unique() if s not in sectors]
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
y = np.arange(len(sectors))
for ax, c in zip(axes[:2], ["US", "JP"]):
    s = sec_share.loc[c].reindex(sectors)
    ax.barh(y + 0.27, s["全体"], 0.27, color="lightgray", label="ユニバース")
    ax.barh(y, s["高収益"], 0.27, color=TIER_COLORS["高収益"], label="高収益")
    ax.barh(y - 0.27, s["低収益"], 0.27, color="#555555", label="低収益")
    ax.set_title(f"{COUNTRY_LABELS[c]}：階層内のセクター構成比")
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend(fontsize=8)
ax = axes[2]
for c, off in (("US", 0.2), ("JP", -0.2)):
    s = sec_hit.loc[c].reindex(sectors)
    ax.barh(y + off, s["高収益"], 0.4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
ax.axvline(1 - TIER_CUTS[1], color="black", ls="--", lw=0.8)
ax.set_title("セクター内で高収益に入る確率（破線=全体平均）")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend()
axes[0].set_yticks(y)
axes[0].set_yticklabels(sectors)
plt.tight_layout()
plt.show()

In [ ]:
# セクター中立の階層（セクター内順位）でのスプレッド：業種構成の違いを除いても日米差は残るか
profile_sec, _ = tier_profile(df, "tier_sector")
cmp = pd.DataFrame({
    "JP−US（基本）": profile["JP−US"],
    "JP−US（セクター中立）": profile_sec["JP−US"],
}).drop(index=DEF_FEATS).sort_values("JP−US（基本）")

fig, ax = plt.subplots(figsize=(9, 0.38 * len(cmp) + 1.5))
yy = np.arange(len(cmp))
ax.barh(yy + 0.2, cmp.iloc[:, 0], 0.4, color="#555555", label=cmp.columns[0])
ax.barh(yy - 0.2, cmp.iloc[:, 1], 0.4, color="#e9a23b", label=cmp.columns[1])
ax.set_yticks(yy)
ax.set_yticklabels([lab(i) for i in cmp.index])
ax.axvline(0, color="black", lw=0.8)
ax.set_title("高−低スプレッドの日米差：業種構成をコントロールしても残るか")
ax.legend()
plt.tight_layout()
plt.show()

## 8. 収益性の持続性

高収益企業は翌年以降も高収益であり続けるか、低収益企業は抜け出せるかを、階層の遷移で比較します。
ランダムなら、各階層に留まる確率はその階層の構成比（既定：高30%・中40%・低30%）です。

In [ ]:
panel = df[["country", "Global_ID", "date", "tier", "q"]].dropna()


def forward_join(p, months):
    f = p[["country", "Global_ID", "date", "tier", "q"]].copy()
    f["date"] = f["date"] - pd.DateOffset(months=months)
    return p.merge(f.rename(columns={"tier": "tier_fwd", "q": "q_fwd"}), on=["country", "Global_ID", "date"], how="inner")


horizons = [12, 24, 36, 48, 60]
stay = {}
for h in horizons:
    m = forward_join(panel, h)
    for t in ["高収益", "低収益"]:
        stay[(t, h)] = (m[m["tier"] == t].assign(s=lambda x: x["tier_fwd"] == t)
                        .groupby(["country", "date"])["s"].mean().groupby("country").mean())
stay = pd.DataFrame(stay).T
trans12 = forward_join(panel, 12)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for c in ["US", "JP"]:
    for t, ls in (("高収益", "-"), ("低収益", ":")):
        s = stay.loc[t, c]
        axes[0].plot(s.index, s.values, marker="o", ls=ls, color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}:{t}")
axes[0].axhline(1 - TIER_CUTS[1], color="gray", ls="--", lw=1, label="ランダム（高）")
axes[0].set_xlabel("経過月数")
axes[0].set_title("同じ階層に留まる確率")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend(fontsize=8)

for ax, c in zip(axes[1:], ["US", "JP"]):
    sub = trans12[trans12["country"] == c]
    t = pd.crosstab(sub["tier"], sub["tier_fwd"], normalize="index").reindex(index=TIERS[::-1], columns=TIERS)
    ax.imshow(t.values, cmap="Blues", vmin=0, vmax=0.9)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.0%}", ha="center", va="center",
                    color="white" if t.values[i, j] > 0.5 else "black")
    ax.set_xticks(range(3)); ax.set_xticklabels(TIERS)
    ax.set_yticks(range(3)); ax.set_yticklabels(TIERS[::-1])
    ax.set_xlabel("12か月後"); ax.set_ylabel("当月")
    ax.set_title(f"{COUNTRY_LABELS[c]}：12か月後の遷移確率")
    ax.grid(False)
plt.tight_layout()
plt.show()
stay.rename(columns=COUNTRY_LABELS)

In [ ]:
# 分位で見た持続性：当月の分位ごとの「12か月後の平均分位」（45度線に近いほど持続的、平らなほど平均回帰）
fwd_q = trans12.groupby(["country", "q"])["q_fwd"].mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for c in ["US", "JP"]:
    s = fwd_q.loc[c]
    axes[0].plot(s.index, s.values, marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].plot([1, N_QUANTILES], [1, N_QUANTILES], color="gray", ls="--", lw=0.8, label="完全持続")
axes[0].axhline((N_QUANTILES + 1) / 2, color="gray", ls=":", lw=0.8, label="完全な平均回帰")
axes[0].set_xlabel("当月の分位"); axes[0].set_ylabel("12か月後の平均分位")
axes[0].set_title("分位の持続性")
axes[0].legend()

p12 = (trans12[trans12["tier"] == "高収益"].assign(s=lambda x: x["tier_fwd"] == "高収益")
       .groupby(["date", "country"])["s"].mean().unstack())
for c in ["US", "JP"]:
    axes[1].plot(p12.index, p12[c].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[1].set_title("12か月後も高収益に留まる確率の推移（12M平均）")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[1].legend()
plt.tight_layout()
plt.show()

### 8-2. 指数からの除外を含めた行き先

上の遷移確率は「12か月後もデータがある銘柄」だけで計算しているため、指数から除外された銘柄が抜け落ち、残存率が高めに出る可能性があります。
ここでは行き先を **高・中・低収益 / 指数内だがデータなし（セクター変更・指標欠損など）/ 指数から除外** の5つに分けて見ます。

In [ ]:
OUTCOMES = ["高収益", "中収益", "低収益", "指数内・データなし", "指数から除外"]
OUT_COLORS = {"高収益": TIER_COLORS["高収益"], "中収益": TIER_COLORS["中収益"], "低収益": TIER_COLORS["低収益"],
              "指数内・データなし": "#bcd4e6", "指数から除外": "#34495e"}

univ = raw[["country", "Global_ID", "year_month"]].copy()
univ["date"] = pd.to_datetime(univ["year_month"].astype(int).astype(str), format="%Y%m")
univ = univ.drop_duplicates(["country", "Global_ID", "date"])[["country", "Global_ID", "date"]]
LAST_DATE = df["date"].max()


def outcome_after(h):
    base = df.dropna(subset=["tier"])[["country", "Global_ID", "date", "tier"]]
    base = base[base["date"] + pd.DateOffset(months=h) <= LAST_DATE]
    fut = df.dropna(subset=["tier"])[["country", "Global_ID", "date", "tier"]].rename(columns={"tier": "tier_fwd"})
    fut["date"] = fut["date"] - pd.DateOffset(months=h)
    inu = univ.assign(in_univ=True)
    inu["date"] = inu["date"] - pd.DateOffset(months=h)
    m = base.merge(fut, on=["country", "Global_ID", "date"], how="left").merge(
        inu, on=["country", "Global_ID", "date"], how="left")
    m["outcome"] = np.select([m["tier_fwd"].notna(), m["in_univ"].notna()],
                             [m["tier_fwd"], "指数内・データなし"], "指数から除外")
    return m


OUT_H = [12, 36, 60]
out_share = {}
for h in OUT_H:
    m = outcome_after(h)
    sh = (m.groupby(["country", "tier", "date"])["outcome"].value_counts(normalize=True)
          .unstack().reindex(columns=OUTCOMES).fillna(0).groupby(["country", "tier"]).mean())   # 月ごとの比率を時系列平均
    out_share[h] = sh
out_share = pd.concat(out_share, names=["h"])

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, start in zip(axes, ["高収益", "低収益"]):
    rows_ = [(h, c) for c in ["US", "JP"] for h in OUT_H]
    labels_ = [f"{COUNTRY_LABELS[c]}:{h // 12}年後" for h, c in rows_]
    left = np.zeros(len(rows_))
    for o in OUTCOMES:
        v = np.array([out_share.loc[(h, c, start), o] for h, c in rows_])
        ax.barh(labels_, v, left=left, color=OUT_COLORS[o], label=o)
        for j, vv in enumerate(v):
            if vv > 0.06:
                ax.text(left[j] + vv / 2, j, f"{vv:.0%}", ha="center", va="center", fontsize=8,
                        color="white" if o in ("指数から除外",) else "black")
        left += v
    ax.invert_yaxis()
    ax.set_title(f"当初「{start}」だった企業の行き先")
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(1.0, -0.08))
plt.tight_layout()
plt.show()

out_share.xs("高収益", level="tier").rename(index=COUNTRY_LABELS, level="country").style.format("{:.1%}")

In [ ]:
# 「高収益に残る確率」：データがある銘柄のみ（従来）vs 除外・欠損も分母に含める
fig, ax = plt.subplots(figsize=(10, 4))
for c in ["US", "JP"]:
    cond, uncond = [], []
    for h in horizons:
        m = outcome_after(h)
        m = m[m["tier"] == "高収益"]
        g = m.groupby("date")
        uncond.append(g.apply(lambda x: (x["outcome"] == "高収益").mean()).mean())
        mm = m[m["outcome"].isin(TIERS)]
        cond.append(mm.groupby("date").apply(lambda x: (x["outcome"] == "高収益").mean()).mean())
    ax.plot(horizons, cond, marker="o", ls="--", color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}：データがある銘柄のみ")
    ax.plot(horizons, uncond, marker="o", color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}：除外・欠損も含む")
ax.axhline(1 - TIER_CUTS[1], color="gray", ls=":", lw=1, label="ランダム")
ax.set_xlabel("経過月数")
ax.set_title("高収益に残る確率：除外銘柄を含めるとどれだけ下がるか")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(fontsize=8)
plt.show()

### 8-3. 高収益は何年続くか（生存分析）

実績の収益性は年1回の決算で更新されるため、毎年 `SNAPSHOT_MONTH` 月時点の階層で年次パネルを作り、
**新たに高収益に入った企業が、何年連続で高収益に留まるか** を Kaplan-Meier 法で推計します。

- 対象：前年に「中収益・低収益」で観測され、当年に「高収益」になった企業（開始時点が分かる期間のみ）
- 終了（イベント）：翌年に中・低収益へ落ちた
- 打ち切り：指数からの除外・データ欠損・サンプル期間の終わり（＝その後どうなったか分からない）
- 比較のため、**低収益から抜け出すまでの年数** も同じ方法で推計

In [ ]:
SNAPSHOT_MONTH = 6     # 年次パネルを作る月（米国12月決算・日本3月決算の実績がともに反映されている時期）


def build_spells(target):
    ann = df[(df["date"].dt.month == SNAPSHOT_MONTH)].dropna(subset=["tier"])[["country", "Global_ID", "date", "tier"]].copy()
    ann["year"] = ann["date"].dt.year
    ann = ann.sort_values(["country", "Global_ID", "year"])
    g = ann.groupby(["country", "Global_ID"])
    ann["is_t"] = ann["tier"] == target
    ann["prev_year"] = g["year"].shift()
    ann["prev_t"] = g["is_t"].shift()
    ann["next_year"] = g["year"].shift(-1)
    consecutive = ann["prev_year"] == ann["year"] - 1
    ann["start"] = ann["is_t"] & ~(consecutive & (ann["prev_t"] == True))
    ann["spell"] = ann.groupby(["country", "Global_ID"])["start"].cumsum()
    sp = ann[ann["is_t"]].groupby(["country", "Global_ID", "spell"]).agg(
        start=("year", "min"), end=("year", "max"), n=("year", "size"),
        next_year=("next_year", "last"))
    # 開始時点が分かるもの：開始の前年に同じ企業が別の階層で観測されている
    first = ann[ann["start"]].set_index(["country", "Global_ID", "spell"])
    sp["known_start"] = (first["prev_year"] == first["year"] - 1).reindex(sp.index).fillna(False).astype(bool)
    sp = sp[sp["known_start"] & (sp["n"] == sp["end"] - sp["start"] + 1)]
    # イベント＝翌年に観測され、別の階層になった
    max_year = ann["year"].max()
    sp["event"] = sp["next_year"] == sp["end"] + 1
    sp["censor_reason"] = np.where(sp["event"], "", np.where(sp["end"] == max_year, "期間終了", "除外・欠損"))
    return sp.reset_index()


def kaplan_meier(dur, event):
    dur, event = np.asarray(dur), np.asarray(event, bool)
    times = np.arange(1, dur.max() + 1)
    S, out = 1.0, [(0, 1.0)]
    for t in times:
        n_risk = (dur >= t).sum()
        d = ((dur == t) & event).sum()
        if n_risk > 0:
            S *= 1 - d / n_risk
        out.append((t, S))
    return pd.Series(dict(out))


km_tbl = {}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3), sharey=True)
for ax, target, title in zip(axes, ["高収益", "低収益"], ["高収益に留まる確率", "低収益に留まる確率（＝抜け出せない確率）"]):
    sp = build_spells(target)
    for c in ["US", "JP"]:
        s = sp[sp["country"] == c]
        if s.empty:
            continue
        km_ = kaplan_meier(s["n"], s["event"])
        ax.step(km_.index, km_.values, where="post", color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}（n={len(s)}）")
        med = km_[km_ <= 0.5].index.min()
        km_tbl[(target, COUNTRY_LABELS[c])] = {
            "1年後": km_.get(1, np.nan), "2年後": km_.get(2, np.nan), "3年後": km_.get(3, np.nan),
            "5年後": km_.get(5, np.nan), "中央値（年）": med if pd.notna(med) else np.nan,
            "スペル数": len(s), "打ち切り：除外・欠損": (s["censor_reason"] == "除外・欠損").mean(),
            "打ち切り：期間終了": (s["censor_reason"] == "期間終了").mean()}
    ax.axhline(0.5, color="gray", ls=":", lw=0.8)
    ax.set_xlim(0, 10)
    ax.set_xlabel("年数")
    ax.set_title(f"{title}（Kaplan-Meier, 毎年{SNAPSHOT_MONTH}月時点）")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend()
plt.tight_layout()
plt.show()

km_tbl = pd.DataFrame(km_tbl).T
km_tbl.style.format({"1年後": "{:.0%}", "2年後": "{:.0%}", "3年後": "{:.0%}", "5年後": "{:.0%}", "中央値（年）": "{:.0f}",
                     "スペル数": "{:.0f}", "打ち切り：除外・欠損": "{:.0%}", "打ち切り：期間終了": "{:.0%}"})

### 8-4. 高収益であり続ける企業の特徴

当月「高収益」の企業を、12か月後も **高収益に残った企業** と **中・低収益に落ちた企業** に分け、
当月時点の特徴（高収益企業の中で標準化したzスコア）の差を比べます（指数除外・欠損は除く）。
収益性の水準そのもの（閾値からどれだけ上か）も含めているので、「単に余裕を持って高収益だった」効果と比べて読んでください。

In [ ]:
m12 = outcome_after(12)
hi = m12[(m12["tier"] == "高収益") & m12["outcome"].isin(TIERS)][["country", "Global_ID", "date", "outcome"]]
hi["stay"] = (hi["outcome"] == "高収益").astype(float)
sub = df.merge(hi[["country", "Global_ID", "date", "stay"]], on=["country", "Global_ID", "date"], how="inner")

feats_p = ["pct"] + [f for f in PROFILE_FEATS if f not in ("roe_estimate",)]
z = cs_zscore(sub, feats_p)
z[["country", "date", "stay"]] = sub[["country", "date", "stay"]]
gm = z.groupby(["country", "date", "stay"])[feats_p].mean()
diff = gm.xs(1.0, level="stay") - gm.xs(0.0, level="stay")
pers = pd.DataFrame({
    "US_差": diff.loc["US"].mean(), "US_t": diff.loc["US"].agg(nw_tstat),
    "JP_差": diff.loc["JP"].mean(), "JP_t": diff.loc["JP"].agg(nw_tstat),
})
LABELS.setdefault("pct", "収益性パーセンタイル（水準）")
pers = pers.sort_values("US_差")

fig, ax = plt.subplots(figsize=(9, 0.36 * len(pers) + 1.6))
y = np.arange(len(pers))
for c, off in (("US", 0.2), ("JP", -0.2)):
    bars = ax.barh(y + off, pers[f"{c}_差"], 0.4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    for b, t in zip(bars, pers[f"{c}_t"]):
        if abs(t) < 2:
            b.set_alpha(0.35)
ax.set_yticks(y)
ax.set_yticklabels([lab(i) for i in pers.index])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("残った企業 − 落ちた企業（高収益企業内のzスコア差）")
ax.set_title("12か月後も高収益に残る企業の特徴（薄い棒は|t|<2）")
ax.legend()
plt.tight_layout()
plt.show()
print("12か月後も高収益に残った割合（除外・欠損を除く）:",
      {COUNTRY_LABELS[c]: f"{v:.0%}" for c, v in hi.groupby("country")["stay"].mean().items()})

### 8-5. 収益性はどれくらいの速さで平均に戻るか

各国・各月で、収益性パーセンタイルの「当月」と「h か月後」の相関（順位の自己相関）を計算します。
1年後の相関 ρ から、差が半分になるまでの期間（**半減期** = ln(0.5)/ln(ρ) 年）を求めます。

In [ ]:
pp = df.dropna(subset=["pct"])[["country", "Global_ID", "date", "pct"]]
ac = {}
for h in [12, 24, 36, 48, 60]:
    f = pp.rename(columns={"pct": "pct_fwd"}).copy()
    f["date"] = f["date"] - pd.DateOffset(months=h)
    m = pp.merge(f, on=["country", "Global_ID", "date"])
    ac[h] = m.groupby(["country", "date"])[["pct", "pct_fwd"]].corr().xs("pct", level=2)["pct_fwd"]
ac = pd.DataFrame(ac)

half_life = {}
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for c in ["US", "JP"]:
    a = ac.loc[c]
    axes[0].plot(a.columns, a.mean().values, marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    rho = a[12].mean()
    half_life[COUNTRY_LABELS[c]] = np.log(0.5) / np.log(rho) if 0 < rho < 1 else np.nan
    axes[1].plot(a.index, a[12].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c],
                 label=f"{COUNTRY_LABELS[c]}（平均 {rho:.2f}, 半減期 {half_life[COUNTRY_LABELS[c]]:.1f}年）")
axes[0].set_xlabel("経過月数")
axes[0].set_title("収益性パーセンタイルの自己相関（減衰）")
axes[0].legend()
axes[1].set_title("1年後の自己相関の推移（12M平均）")
axes[1].legend()
plt.tight_layout()
plt.show()
pd.Series(half_life, name="半減期（年）").to_frame()

## 9〜12. 分位別の絶対水準

5-2章は zスコア（自国内の相対位置）でしたが、ここでは **生の値の中央値** を分位ごとに描きます（月次中央値の時系列平均）。
実際の水準（例：総還元性向が何%か）を日米で比べるときに使います。

In [ ]:
def plot_quantile_levels(data, cols, title, ncols=3):
    cols = [c for c in cols if c in data.columns]
    prof = (data.dropna(subset=["q"]).groupby(["country", "date", "q"])[cols].median()
            .groupby(["country", "q"]).mean())
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.3 * nrows), squeeze=False)
    for ax, col in zip(axes.flat, cols):
        for c in ["US", "JP"]:
            s = prof.loc[c, col]
            ax.plot(s.index, s.values, marker="o", ms=4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
        for t0 in TIER_CUTS:
            ax.axvline(t0 * N_QUANTILES + 0.5, color="gray", lw=0.6, ls=":")
        ax.set_xticks(range(1, N_QUANTILES + 1))
        ax.set_title(lab(col))
    for ax in axes.flat[len(cols):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle(f"{title}（{lab(HIGH_PROF_VAR)}分位別の中央値, 点線は3階層の境界）", y=1.01)
    plt.tight_layout()
    plt.show()
    out = prof[cols].unstack("country")
    out.columns = [f"{lab(a)}:{COUNTRY_LABELS[b]}" for a, b in out.columns]
    return out


def plot_tier_ts(data, cols, title):
    """階層別中央値の時系列（12M平均）"""
    cols = [c for c in cols if c in data.columns]
    med = data.dropna(subset=["tier"]).groupby(["date", "country", "tier"])[cols].median()
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 3.8), squeeze=False)
    for ax, col in zip(axes.flat, cols):
        for c in ["US", "JP"]:
            for t in TIERS:
                try:
                    s = med[col].xs((c, t), level=("country", "tier")).rolling(12, min_periods=6).mean()
                except KeyError:
                    continue
                ax.plot(s.index, s, color=COUNTRY_COLORS[c], ls=TIER_STYLE[t], label=f"{COUNTRY_LABELS[c]}:{t}")
        ax.set_title(lab(col))
    axes.flat[0].legend(fontsize=8, ncol=2)
    fig.suptitle(title + "（中央値, 12M平均）", y=1.02)
    plt.tight_layout()
    plt.show()

### 9. 利益の質・予想収益性

In [ ]:
_ = plot_quantile_levels(
    df,
    ["operating_cf_to_assets", "accrual", "roa_actual", "roe_estimate", "roe_change_est", "profitability_factor"],
    "利益の質・予想収益性",
)

- **アクルーアル**（ROA − 営業CF/総資産）が高い＝利益がキャッシュを伴っていない。上位分位でこれが高い国は「利益の質」に注意。
- **予想ROE − 実績ROE** が上位でマイナス・下位でプラス＝アナリストが平均回帰を予想。傾きの強さを日米で比較。

### 10. 株主還元：稼いだ利益をどれだけ株主に返しているか

配当か自社株買いかの手段は問わず、**総還元**で見ます。

- **総還元利回り**（配当＋自社株買い）÷ 時価総額：投資家から見た還元の大きさ
- **総還元性向** ＝ 総還元利回り ÷ E/P ＝（配当＋自社株買い）÷ 純利益：**稼いだ利益のうち何割を返しているか**
- **純還元**は株式発行（希薄化）を差し引いたもの

総還元性向は E/P>0（黒字）の企業のみで計算しています。100%超は利益以上を還元（手元資金の取り崩しや借入）していることを表します。

In [ ]:
_ = plot_quantile_levels(
    df,
    ["total_payout_ratio", "net_payout_ratio", "gross_payout_yield", "net_payout_yield"],
    "総還元", ncols=4,
)

In [ ]:
# 階層別の総還元性向の構成：各階層の企業は利益の何割を返しているか（全期間プール）
hp = df.dropna(subset=["tier", "total_payout_ratio"])
bins = [-np.inf, 0.3, 0.5, 0.8, 1.0, np.inf]
names = ["30%未満", "30-50%", "50-80%", "80-100%", "100%以上"]
dist = (hp.assign(b=pd.cut(hp["total_payout_ratio"], bins, labels=names).astype("object"))
        .groupby(["country", "tier"])["b"].value_counts(normalize=True).unstack()
        .reindex(columns=names).reindex([(c, t) for c in ["US", "JP"] for t in TIERS[::-1]]))
rows = [f"{COUNTRY_LABELS[c]}:{t}" for c, t in dist.index]

fig, ax = plt.subplots(figsize=(12, 4))
left = np.zeros(len(dist))
cmap = plt.get_cmap("Greens")
for i, n in enumerate(names):
    v = dist[n].fillna(0).to_numpy()
    ax.barh(rows, v, left=left, color=cmap(0.25 + 0.15 * i), label=n)
    for j, vv in enumerate(v):
        if vv > 0.05:
            ax.text(left[j] + vv / 2, j, f"{vv:.0%}", ha="center", va="center", fontsize=9)
    left += v
ax.invert_yaxis()
ax.set_title("収益階層別の総還元性向の構成")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.show()
dist.rename(index=COUNTRY_LABELS, level=0)

In [ ]:
plot_tier_ts(df, ["total_payout_ratio", "gross_payout_yield"], "収益階層別の総還元の推移")

# 収益性と還元の連動：国×月ごとに『収益性の順位』と『総還元性向の順位』の相関（スピアマン）
#   米国で高く日本で低ければ、米国の方が「稼ぐ企業ほど返す」構造
rk = df.dropna(subset=["pct", "total_payout_ratio"])
rk = rk.assign(r_pay=rk.groupby(["country", "date"])["total_payout_ratio"].rank(pct=True))
corr = rk.groupby(["date", "country"])[["pct", "r_pay"]].corr().xs("pct", level=2)["r_pay"].unstack()

fig, ax = plt.subplots(figsize=(12, 3.8))
for c in ["US", "JP"]:
    ax.plot(corr.index, corr[c].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c],
            label=f"{COUNTRY_LABELS[c]}（全期間平均 {corr[c].mean():+.2f}）")
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"{lab(HIGH_PROF_VAR)}と総還元性向の順位相関（12M平均）")
ax.legend()
plt.show()

### 11. バリュエーション：市場は高収益企業を評価しているか

In [ ]:
_ = plot_quantile_levels(df, ["book_to_price_actual", "earnings_to_price_actual"], "バリュエーション", ncols=2)

# 各階層のB/P ÷ 全体のB/P（中央値どうし、1未満＝割高に評価されている）
bp = df.dropna(subset=["tier", "book_to_price_actual"])
rel = (bp.groupby(["date", "country", "tier"])["book_to_price_actual"].median()
       / bp.groupby(["date", "country"])["book_to_price_actual"].median()).unstack(["country", "tier"])
fig, ax = plt.subplots(figsize=(12, 3.8))
for c in ["US", "JP"]:
    for t in ["高収益", "低収益"]:
        ax.plot(rel.index, rel[(c, t)].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c],
                ls=TIER_STYLE[t], label=f"{COUNTRY_LABELS[c]}:{t}")
ax.axhline(1, color="black", lw=0.8)
ax.set_title("階層のB/P中央値 ÷ 全体のB/P中央値（12M平均）：高収益の線が低いほど市場がプレミアムを付与")
ax.legend(ncol=2)
plt.show()

B/P は「ROE × B/P = E/P」の関係で ROE と機械的に結び付くため、上位分位で B/P が低いのはある程度自然です。
注目点は **プレミアムの大きさの日米差** と **その推移**（例：日本で東証のPBR改善要請後に変化したか）。

### 12. 規模・成長・投資・財務構成

In [ ]:
_ = plot_quantile_levels(
    df,
    ["log_mcap", "sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual", "equity_ratio", "debt_to_assets"],
    "規模・成長・投資・財務構成",
)

## 13. 多変量分析：何が収益性を特徴づけるか（Fama-MacBeth型）

単変量では相関し合う特徴（例：自己資本比率と負債比率、規模と還元）の効果が混ざるため、
各国・各月で被説明変数を自国内で標準化した特徴量に回帰し、係数の時系列平均と Newey-West t値を求めます。

- **(a) 収益性パーセンタイル**（0〜1, 連続）：分位全体を通じた線形の関係
- **(b) 高収益ダミー**（上位階層=1）：高収益企業になる確率への効果（線形確率モデル）

ROE の定義に直接入る純利益率・ROA などは説明変数から外しています（B/P・回転率は ROE と機械的関係があるので解釈注意）。

In [ ]:
FM_FEATS = [c for c in [
    "log_mcap", "equity_ratio", "asset_turnover_actual", "gross_margin_actual",
    "sales_growth_actual", "asset_growth_actual", "book_to_price_actual",
    "gross_payout_yield",
] if c in df.columns]


def fama_macbeth(data, y_col, feats, min_obs=30):
    out = []
    for (c, d), g in data.dropna(subset=[y_col] + feats).groupby(["country", "date"]):
        if len(g) < max(min_obs, len(feats) + 10):
            continue
        X = g[feats]
        X = (X - X.mean()) / X.std()
        if X.isna().any().any():
            continue
        X = np.column_stack([np.ones(len(X)), X.to_numpy()])
        y = g[y_col].astype(float).to_numpy()
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        out.append(pd.Series(beta[1:], index=feats, name=(c, d)))
    coefs = pd.DataFrame(out)
    coefs.index = pd.MultiIndex.from_tuples(coefs.index, names=["country", "date"])
    return pd.DataFrame({
        "coef": coefs.groupby("country").mean().stack(),
        "t": coefs.groupby("country").agg(lambda s: nw_tstat(s)).stack(),
    })


fm = {"(a) 収益性パーセンタイル": fama_macbeth(df, "pct", FM_FEATS),
      "(b) 高収益ダミー": fama_macbeth(df, "is_high", FM_FEATS)}
fm_tbl = pd.concat({k: v.unstack("country") for k, v in fm.items()}, axis=1)
fm_tbl.index = [lab(i) for i in fm_tbl.index]
fm_tbl

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 0.5 * len(FM_FEATS) + 1.8), sharey=True)
base = fm["(a) 収益性パーセンタイル"]
order = base.xs("JP", level="country")["coef"].sub(base.xs("US", level="country")["coef"]).sort_values().index
y = np.arange(len(order))
for ax, (name, res) in zip(axes, fm.items()):
    for c, off in (("US", 0.18), ("JP", -0.18)):
        r = res.xs(c, level="country").reindex(order)
        se = (r["coef"] / r["t"]).abs()
        ax.errorbar(r["coef"], y + off, xerr=1.96 * se, fmt="o", color=COUNTRY_COLORS[c],
                    label=COUNTRY_LABELS[c], capsize=3)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("特徴量1σ上昇あたりの変化")
axes[0].set_yticks(y)
axes[0].set_yticklabels([lab(i) for i in order])
axes[0].legend()
fig.suptitle("月次クロスセクション回帰の係数（95%区間, Newey-West）", y=1.01)
plt.tight_layout()
plt.show()

## 14. 頑健性チェック

### 14-1. 収益性の定義を変える
ROE（基本）／ROA／収益性ファクター／ROE（セクター中立）の4定義で、高−低スプレッドの日米差（JP−US）が同じ方向かを確認します。

In [ ]:
alt_defs = {}
for var in [v for v in ["roe_actual", "roa_actual", "profitability_factor"] if v in df.columns]:
    tmp = df.copy()
    tmp["tier_alt"] = to_tier(pct_rank(tmp, var, ["country", "date"]))
    alt_defs[lab(var)] = tier_profile(tmp, "tier_alt")[0]["JP−US"]
alt_defs[f"{lab(HIGH_PROF_VAR)}（セクター中立）"] = profile_sec["JP−US"]
alt = pd.DataFrame(alt_defs).drop(index=DEF_FEATS)
alt = alt.sort_values(alt.columns[0])
heatmap(alt, "定義別：高−低スプレッドの日米差（JP−US, zスコア）")

### 14-2. 期間を分ける

In [ ]:
per = {}
for name, (y0, y1) in PERIODS.items():
    sub = df[(df["date"].dt.year >= y0) & (df["date"].dt.year <= y1)]
    if sub.empty:
        continue
    r, _ = tier_profile(sub)
    per[f"{name}\n米国"] = r["US_高−低"]
    per[f"{name}\n日本"] = r["JP_高−低"]
per = pd.DataFrame(per)
heatmap(per, "期間別：高−低スプレッド（zスコア）", vmax=1.5, xrot=0)

### 14-3. 階層の区切りを変える

In [ ]:
cut_sets = {"下位20/上位20": (0.2, 0.8), "下位30/上位30（基本）": (0.3, 0.7), "下位40/上位40": (0.4, 0.6)}
res_cut = {}
for name, (a, b) in cut_sets.items():
    tmp = df.copy()
    tmp["tier_alt"] = pd.cut(tmp["pct"], [0, a, b, 1.0], labels=TIERS, include_lowest=True).astype("object").where(tmp["pct"].notna())
    res_cut[name] = tier_profile(tmp, "tier_alt")[0]["JP−US"]
heatmap(pd.DataFrame(res_cut).drop(index=DEF_FEATS).sort_values("下位30/上位30（基本）"),
        "区切り別：高−低スプレッドの日米差（JP−US）")

## 15. まとめ（自動抽出）

以下は上の結果から機械的に抽出した要点です。数値を見ながら解釈を加えてください。

In [ ]:
def describe_findings(top=6):
    lines = []
    p = profile.drop(index=DEF_FEATS).copy()
    p["abs_gap"] = p["JP−US"].abs()
    lines.append(f"■ 高−低スプレッドの日米差が大きい項目（{TIER_DESC}）")
    for f, r in p.sort_values("abs_gap", ascending=False).head(top).iterrows():
        u, j = r["US_高−低"], r["JP_高−低"]
        if np.sign(u) != np.sign(j):
            note = "日米で向きが逆"
        else:
            strong = "日本" if abs(j) > abs(u) else "米国"
            note = f"{strong}の方が収益性との結び付きが強い（{'高収益ほど高い' if u > 0 else '高収益ほど低い'}）"
        lines.append(
            f"  - {lab(f)}: 米国 {u:+.2f} (t={r['US_t']:.1f}) / 日本 {j:+.2f} (t={r['JP_t']:.1f}) → {note}")

    lines.append("\n■ 日米で向きが逆（双方 |t|>2）")
    opp = p[(np.sign(p["US_高−低"]) != np.sign(p["JP_高−低"])) & (p["US_t"].abs() > 2) & (p["JP_t"].abs() > 2)]
    lines += [f"  - {lab(f)}: 米国 {r['US_高−低']:+.2f} / 日本 {r['JP_高−低']:+.2f}" for f, r in opp.iterrows()] or ["  - なし"]

    lines.append(f"\n■ 分位（{N_QUANTILES}分位）に沿った形状が日米で異なる項目")
    sw = shape["形状"].unstack("country").drop(index=DEF_FEATS, errors="ignore")
    diff_shape = sw[sw["US"] != sw["JP"]]
    lines += [f"  - {lab(f)}: 米国「{r['US']}」/ 日本「{r['JP']}」" for f, r in diff_shape.iterrows()] or ["  - なし"]
    nonlin = shape[shape["形状"].str.contains("加速|頭打ち|U字")]
    lines.append("\n■ 線形でない（上位で加速・頭打ち・U字）項目")
    lines += [f"  - {lab(f)}（{COUNTRY_LABELS[c]}）: {r['形状']}（曲率t={r['曲率t']:.1f}）"
              for (f, c), r in nonlin.iterrows() if f not in DEF_FEATS] or ["  - なし"]

    lines.append("\n■ ROE格差の源泉（DuPont分散分解, 全期間平均）")
    for c in share_mean.index:
        s = share_mean.loc[c].drop("その他（近似誤差）")
        lines.append(f"  - {c}: " + ", ".join(f"{k} {v:.0%}" for k, v in s.items()) + f" → 主因は{s.idxmax()}")

    lines.append("\n■ 高収益÷低収益（黒字企業内, DuPont構成要素の倍率）")
    for c in ["米国", "日本"]:
        r = ratio.loc[("高収益÷低収益", c)]
        lines.append(f"  - {c}: " + ", ".join(f"{k} {v:.2f}倍" for k, v in r.items()))

    lines.append("\n■ 持続性（同じ階層に留まる確率）")
    for h in horizons:
        lines.append(f"  - {h}か月後: 高収益 米国 {stay.loc[('高収益', h), 'US']:.0%} / 日本 {stay.loc[('高収益', h), 'JP']:.0%}"
                     f"  |  低収益 米国 {stay.loc[('低収益', h), 'US']:.0%} / 日本 {stay.loc[('低収益', h), 'JP']:.0%}")

    lines.append("\n■ 高収益が続く年数（Kaplan-Meier, 新たに高収益に入った企業）")
    for c in ["米国", "日本"]:
        if ("高収益", c) in km_tbl.index:
            r = km_tbl.loc[("高収益", c)]
            lines.append(f"  - {c}: 1年後 {r['1年後']:.0%} / 3年後 {r['3年後']:.0%} / 5年後 {r['5年後']:.0%}, 中央値 {r['中央値（年）']:.0f}年")
    lines.append("  - 収益性の半減期: " + ", ".join(f"{k} {v:.1f}年" for k, v in half_life.items()))
    pers_x = pers.drop(index=[i for i in DEF_FEATS + ["pct"] if i in pers.index])
    lines.append("  - 高収益に残る企業の特徴（収益性の水準以外, 残った−落ちた, |t|>2）: " + " / ".join(
        f"{COUNTRY_LABELS[c]} " + ", ".join(f"{lab(i)}({v:+.2f})" for i, v in pers_x[pers_x[f'{c}_t'].abs() > 2][f'{c}_差']
                                            .sort_values(key=abs, ascending=False).head(4).items())
        for c in ["US", "JP"]))

    lines.append("\n■ 多変量回帰（収益性パーセンタイル）で有意（|t|>2）な特徴")
    res = fm["(a) 収益性パーセンタイル"]
    for c in ["US", "JP"]:
        r = res.xs(c, level="country")
        sig = r[r["t"].abs() > 2].sort_values("t")
        lines.append(f"  - {COUNTRY_LABELS[c]}: " + (", ".join(f"{lab(i)}({v:+.3f})" for i, v in sig["coef"].items()) or "なし"))
    return "\n".join(lines)


print(f"定義: {lab(HIGH_PROF_VAR)} の国内パーセンタイル（{TIER_DESC}, {N_QUANTILES}分位）  /  "
      f"除外セクター: {EXCLUDE_SECTOR_CODES}  /  期間: {df['date'].min():%Y-%m}〜{df['date'].max():%Y-%m}\n")
print(describe_findings())

### 解釈の観点（検証すべき仮説）

結果を読む際の主なチェックポイントです。上の出力でどれが支持されるかを確認してください。

1. **稼ぎ方の違い**：日本の高収益企業は「マージン型」か「回転率型」か。米国はレバレッジ（自己資本比率の低さ）がROEを押し上げているか
   → 6章（DuPont）と5章の自己資本比率・負債比率
2. **財務の保守性**：日本の高収益企業は自国内でも自己資本比率が高い（キャッシュリッチ）のか。そうならROEを抑える要因になっている可能性
3. **稼いだ利益の還元**：米国の高収益企業は利益の大半（総還元性向が高い）を株主に返し、日本の高収益企業は内部留保に回しているか。
   日本で近年その差が縮まっているか → 10章の総還元性向
4. **市場の評価**：高収益企業へのバリュエーション・プレミアムは日米どちらが大きいか、日本で拡大しているか → 11章
5. **持続性**：日本の高収益企業の方が入れ替わりが激しい（景気敏感・市況依存）か、何年続くか、
   どんな高収益企業が残るか（マージン型か回転率型か、財務が保守的か） → 8章
6. **業種要因**：日米差が業種構成（米国はIT・ヘルスケア、日本は資本財・一般消費財など）で説明されるか → 7章のセクター中立比較
7. **成長との関係**：高収益企業は成長企業でもあるのか（米国）、成熟企業なのか（日本） → 12章
8. **線形性**：特徴は収益性に比例して変わるのか、上位（高収益）だけが特殊なのか → 5-2章の形状判定